# Chapitre 2 - Cours 5 : triplets, SPARQL et Datalog

Le cours 4 a présenté le graphe de propriétés et Cypher. Ce cours explore l'autre grande
famille, les **magasins de triplets** et leur langage SPARQL, puis **Datalog**, le plus
ancien et le plus expressif des langages de requêtes récursives, qui fonde tous les
autres. Il couvre les sections *Triple-Stores and SPARQL* (*The semantic web*, *The RDF
data model*, *The SPARQL query language*, encadré *Graph Databases Compared to the
Network Model*) et *The Foundation: Datalog*.

**Plan.**

1. Tout est triplet : sujet, prédicat, objet.
2. Le Web sémantique et le modèle RDF.
3. SPARQL : motifs et chemins de propriétés.
4. Les bases de graphes face au modèle réseau.
5. Datalog : des faits et des règles.
6. Évaluer Datalog : naïf contre semi-naïf.

In [1]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import datamodels as dm
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 100)
rng = np.random.default_rng(0)
fmt = lambda v: f"{v:,.0f}".replace(",", " ")

## 1. Tout est triplet : sujet, prédicat, objet (section *Triple-Stores and SPARQL*)

Le modèle du **magasin de triplets** dit presque la même chose que le graphe de
propriétés, avec d'autres mots. Toute l'information y est rangée en énoncés très
simples à trois parties, **(sujet, prédicat, objet)** : dans (Inès, aime, la poutine),
Inès est le sujet, « aime » le prédicat (le verbe) et la poutine l'objet.

Le sujet est toujours un sommet. L'objet est l'une de deux choses :

- une **valeur primitive** (chaîne, nombre) : prédicat et objet jouent alors le rôle
  de la clé et de la valeur d'une **propriété** du sujet. (ines, birthYear, 1988)
  équivaut au sommet `ines` avec la propriété `{"birthYear": 1988}` ;
- un **autre sommet** : le prédicat est alors une **arête**, de la queue (le sujet) vers
  la tête (l'objet). (ines, livesIn, montreal) est l'arête `LIVES_IN`.

`dm.graph_to_triples` traduit notre graphe de propriétés en triplets ; voici la
sérialisation au format **Turtle** (un sous-ensemble de Notation3), où `_:nom` désigne un
sommet anonyme, `a` abrège « est de type », et le point-virgule enchaîne plusieurs
énoncés sur le même sujet.

In [2]:
g = dm.example_graph()
ts = dm.graph_to_triples(g)
print(f"{len(ts)} triplets\n")
print(ts.to_turtle())

70 triplets

@prefix : <urn:example:>.
_:europe a :Location; :name "Europe"; :type "continent".
_:amnord a :Location; :name "Amérique du Nord"; :type "continent".
_:asie a :Location; :name "Asie"; :type "continent".
_:france a :Location; :name "France"; :type "pays"; :within _:europe.
_:canada a :Location; :name "Canada"; :type "pays"; :within _:amnord.
_:japon a :Location; :name "Japon"; :type "pays"; :within _:asie.
_:aura a :Location; :name "Auvergne-Rhône-Alpes"; :type "région"; :within _:france.
_:bretagne a :Location; :name "Bretagne"; :type "région"; :within _:france.
_:rhone a :Location; :name "Rhône"; :type "département"; :within _:aura.
_:lyon a :Location; :name "Lyon"; :type "ville"; :within _:rhone.
_:quebec a :Location; :name "Québec"; :type "province"; :within _:canada.
_:montreal a :Location; :name "Montréal"; :type "ville"; :within _:quebec.
_:kansai a :Location; :name "Kansai"; :type "région"; :within _:japon.
_:osaka a :Location; :name "Osaka"; :type "ville"; :within 

Un magasin de triplets doit répondre à des motifs où n'importe lequel des trois termes
peut être inconnu : « tous les prédicats de Lyon » `(lyon, ?, ?)`, « tous les sujets nés à
Lyon » `(?, bornIn, lyon)`, « tout ce qui pointe vers Montréal » `(?, ?, montreal)`. Notre
`TripleStore` tient pour cela **trois index** (SPO, POS, OSP) : chaque forme de motif
trouve un index dont le préfixe correspond aux termes connus.

In [3]:
motifs = [("_:lyon", None, None), ("_:lyon", "within", None), (None, "bornIn", "_:lyon"),
          (None, "bornIn", None), (None, None, "_:montreal"), ("_:ines", None, "_:montreal")]
def index_used(s, p, o):
    return "SPO" if s is not None else ("POS" if p is not None else ("OSP" if o is not None else "parcours complet"))
show = lambda t: "?" if t is None else t
pd.DataFrame({"motif": [f"({show(s)}, {show(p)}, {show(o)})" for s, p, o in motifs],
              "index utilisé": [index_used(*m) for m in motifs],
              "réponses": [", ".join(f"({a}, {b}, {c})" for a, b, c in ts.match(*m))[:85] for m in motifs]})

,motif,index utilisé,réponses
0,"(_:lyon, ?, ?)",SPO,"(_:lyon, a, Location), (_:lyon, name, Lyon), (_:lyon, type, ville), (_:lyon, within,"
1,"(_:lyon, within, ?)",SPO,"(_:lyon, within, _:rhone)"
2,"(?, bornIn, _:lyon)",POS,"(_:ines, bornIn, _:lyon)"
3,"(?, bornIn, ?)",POS,"(_:ines, bornIn, _:lyon), (_:kenji, bornIn, _:osaka), (_:lea, bornIn, _:bretagne)"
4,"(?, ?, _:montreal)",OSP,"(_:ines, livesIn, _:montreal), (_:kenji, livesIn, _:montreal)"
5,"(_:ines, ?, _:montreal)",SPO,"(_:ines, livesIn, _:montreal)"


> **À retenir.** Un magasin de triplets range tout en (sujet, prédicat, objet). Un objet
> littéral fait du triplet une propriété, un objet sommet en fait une arête : le modèle
> est équivalent au graphe de propriétés. Trois index (SPO, POS, OSP) servent tous les
> motifs.